In [ ]:
from __future__ import annotations
%cd ../..

In [ ]:
!ls

In [ ]:
track_chromas = {}

In [ ]:
from pathlib import Path
import numpy as np
import sys

sys.path.insert(0, str(Path(".").resolve()))

from mgs_evals.harmonic.hc import harmonic_compatibility
from mgs_evals.harmonic.tc import tonal_coherence, tc_short_clip
from mgs_evals.harmonic.profiles import load_profiles, build_key_templates
from mgs_evals.harmonic._preprocess import track_stems_bar_chroma


REPO_ROOT = Path(".").resolve()
SLAKH_TEST_DIR = REPO_ROOT / "data" / "slakh2100" / "test"
SLAKH_TEST_DIR_15 = REPO_ROOT / "data" / "slakh2100" / "segmented_test_15s"
MSGLD_GEN_DIR  = REPO_ROOT / "inference" / "MSG-LD" / "gen"
MSDM_GEN_DIR   = REPO_ROOT / "inference" / "MSDM" / "msdm_total_generation_bundle" / "gen_tracks"
MSLDM_GEN_DIR  = REPO_ROOT / "inference" / "MSLDM" / "gen"
MGELDM_GEN_DIR = None # not applicable

PITCHED_STEMS = ["guitar", "bass", "piano"]
PROFILE_SET = "ks"
SEGMENT_SIZE = 4
HOP_LENGTH = 512
MAX_TRACKS = 200  # increase for reliable statistics (set to None for all)

PAIRS = [f"hc_{a}_{b}" for i, a in enumerate(PITCHED_STEMS) for b in PITCHED_STEMS[i+1:]]


def _load_track_stems(track_dir: Path) -> dict[str, np.ndarray] | None:
    """Load bar-level chroma for pitched stems, using drums for shared beat tracking."""
    stem_files = {
        stem: str(track_dir / f"{stem}.wav")
        for stem in PITCHED_STEMS
        if (track_dir / f"{stem}.wav").exists()
    }
    if len(stem_files) < 2:
        return None

    drums_path = track_dir / "drums.wav"
    beat_ref = str(drums_path) if drums_path.exists() else None

    return track_stems_bar_chroma(stem_files, hop_length=HOP_LENGTH, beat_ref_file=beat_ref)


def _eval_folder(
    folder: Path,
    templates: np.ndarray,
    max_tracks: int | None = MAX_TRACKS,
) -> dict:
    track_dirs = sorted(d for d in folder.iterdir() if d.is_dir())
    if max_tracks is not None:
        track_dirs = track_dirs[:max_tracks]

    hc_vals: list[float] = []
    tc_vals: list[float] = []
    pair_vals: dict[str, list[float]] = {p: [] for p in PAIRS}

    for td in track_dirs:
        stems = _load_track_stems(td)
        if stems is None:
            continue

        hc_r = harmonic_compatibility(stems)
        if not np.isnan(hc_r["hc"]):
            hc_vals.append(hc_r["hc"])
        for p in PAIRS:
            v = hc_r.get(p, float("nan"))
            if not np.isnan(v):
                pair_vals[p].append(v)

        n_bars = min(c.shape[1] for c in stems.values())
        if n_bars < SEGMENT_SIZE:
            tc_score = tc_short_clip(stems, templates)
        else:
            tc_score = tonal_coherence(stems, templates, SEGMENT_SIZE)["tc"]

        if not np.isnan(tc_score):
            tc_vals.append(tc_score)

    return {"hc": hc_vals, "tc": tc_vals, "pairs": pair_vals}


def _random_baseline(
    templates: np.ndarray,
    n_tracks: int = 200,
    n_bars: int = 8,
    seed: int = 0,
) -> dict:
    rng = np.random.default_rng(seed)
    hc_vals, tc_vals = [], []
    pair_vals: dict[str, list[float]] = {p: [] for p in PAIRS}
    for _ in range(n_tracks):
        stems = {
            name: rng.dirichlet(np.ones(12), size=n_bars).T
            for name in PITCHED_STEMS
        }
        hc_r = harmonic_compatibility(stems)
        if not np.isnan(hc_r["hc"]):
            hc_vals.append(hc_r["hc"])
        for p in PAIRS:
            v = hc_r.get(p, float("nan"))
            if not np.isnan(v):
                pair_vals[p].append(v)
        tc_r = tonal_coherence(stems, templates, SEGMENT_SIZE)
        if not np.isnan(tc_r["tc"]):
            tc_vals.append(tc_r["tc"])
    return {"hc": hc_vals, "tc": tc_vals, "pairs": pair_vals}


def _fmt(vals: list[float]) -> str:
    if not vals:
        return "N/A"
    return f"{float(np.mean(vals)):8.4f}"


def _print_table(rows: list[tuple[str, dict]]):
    pair_cols = [p.replace("hc_", "") for p in PAIRS]
    pair_header = "  ".join(f"{c:>14}" for c in pair_cols)
    print(f"{'Source':<30}  {'HC':>8}  {pair_header}  {'TC':>8}  {'n_hc':>6}  {'n_tc':>6}")
    print()
    for label, data in rows:
        pair_str = "  ".join(_fmt(data["pairs"].get(p, [])) for p in PAIRS)
        print(f"{label:<30}  {_fmt(data['hc'])}  {pair_str}  {_fmt(data['tc'])}  {len(data['hc']):>6}  {len(data['tc']):>6}")


In [ ]:
major, minor = load_profiles(PROFILE_SET)
templates = build_key_templates(major, minor)

rows = []

print(f"Evaluating on {MAX_TRACKS} tracks for each model")

if SLAKH_TEST_DIR_15.exists():
    print(f"Evaluating Slakh 15s")
    rows.append(("Slakh2100 15s (real)", _eval_folder(SLAKH_TEST_DIR_15, templates)))
else:
    print(f"Slakh 15s not found at {SLAKH_TEST_DIR_15}")

if MSGLD_GEN_DIR.exists():
    print(f"Evaluating MSG-LD")
    rows.append(("MSG-LD generated", _eval_folder(MSGLD_GEN_DIR, templates)))
else:
    print(f"MSG-LD gen not found at {MSGLD_GEN_DIR}")

if MSDM_GEN_DIR.exists():
    print(f"Evaluating MSDM")
    rows.append(("MSDM generated", _eval_folder(MSDM_GEN_DIR, templates)))
else:
    print(f"MSDM gen not found at {MSDM_GEN_DIR}")

if MSLDM_GEN_DIR.exists():
    print(f"Evaluating MSLDM")
    rows.append(("MSLDM generated", _eval_folder(MSLDM_GEN_DIR, templates)))
else:
    print(f"MSLDM gen not found at {MSLDM_GEN_DIR}")

print("Skipping MGE-LDM: outputs are mix-only WAVs, no stem separation available.")

print("Computing random baseline")
rows.append(("Random baseline", _random_baseline(templates, n_tracks=MAX_TRACKS or 200)))

_print_table(rows)

In [ ]:
for label, data in rows:
    print(f"{label}: HC mean={np.mean(data['hc']):.4f} (n={len(data['hc'])}), TC mean={np.mean(data['tc']):.4f} (n={len(data['tc'])})")